In [21]:
import pandas as pd
import json
import os
from scipy.stats import pointbiserialr

In [2]:
models = ['0.5B-Qwen25','1.5B-Qwen25','3B-Qwen25','7B-Qwen25','14B-Qwen25','32B-Qwen25',"72B-Qwen25"]
benchmarks = ['USMLE_STEP_1','USMLE_STEP_2','USMLE_STEP_3','MedQA','MedExpQA']

In [14]:
def pb_analysis(metric):    
    baseline_response_path = "/projectnb/vkolagrp/yiliu/QA_pipeline/Results/baseline_metric"
    rows = []
    for b in benchmarks:
        model_avg_r = 0
        for m in models:
            p = f"{baseline_response_path}/{m}_{b}_overlap.csv"
            if not os.path.exists(p): continue
            df = pd.read_csv(p)
            df['acc'] = df['acc'].astype(int)
            r, pval = pointbiserialr(df['acc'], df[metric])
            model_avg_r += r
            rows.append({'model': m, 'benchmark': b, 'r': round(r,4), 'p': round(pval,4)})
        # print(f"{b} average correlation: {model_avg_r/7}")
    res = pd.DataFrame(rows)
    print(res.pivot(index='model', columns='benchmark', values='r').to_string())

In [15]:
pb_analysis('bleu4')

benchmark    MedExpQA   MedQA  USMLE_STEP_1  USMLE_STEP_2  USMLE_STEP_3
model                                                                  
0.5B-Qwen25    0.0763 -0.0186       -0.1128        0.0192       -0.0747
1.5B-Qwen25    0.1445  0.0144       -0.0995        0.0564        0.0601
14B-Qwen25     0.0768  0.0275        0.0325        0.0094        0.0050
32B-Qwen25     0.1456 -0.0249        0.1224        0.0373        0.0116
3B-Qwen25      0.1479  0.0703        0.0405       -0.0513        0.0721
72B-Qwen25     0.1901 -0.0215        0.0870       -0.0395        0.0175
7B-Qwen25      0.0121  0.0004        0.0824       -0.1089        0.0293


In [16]:
pb_analysis('bleu1')

benchmark    MedExpQA   MedQA  USMLE_STEP_1  USMLE_STEP_2  USMLE_STEP_3
model                                                                  
0.5B-Qwen25    0.0569  0.0486       -0.0493       -0.0064       -0.0133
1.5B-Qwen25    0.0455  0.0265        0.1212       -0.1512        0.0700
14B-Qwen25     0.0763 -0.0286        0.1827       -0.0579        0.0510
32B-Qwen25     0.0380 -0.0464        0.1412        0.1639        0.0975
3B-Qwen25      0.0229 -0.0203        0.0513       -0.1118        0.0873
72B-Qwen25     0.0295 -0.0823        0.1030       -0.0419        0.1087
7B-Qwen25     -0.0075 -0.0511        0.0878        0.0078        0.0533


In [40]:
def concat_pb_analysis(metric):     
    for b in benchmarks:
        all_dfs = []
        for m in models:
            p = f"{baseline_response_path}/{m}_{b}_overlap.csv"
            if not os.path.exists(p): continue
            df = pd.read_csv(p)[['acc', metric]].copy()
            df['acc'] = df['acc'].astype(int)
            all_dfs.append(df)
        if not all_dfs: continue
        combined = pd.concat(all_dfs, ignore_index=True)
        r, pval = pointbiserialr(combined['acc'], combined[metric])
        # print(f"{b:20s}: r={r:.4f}, p={pval:.4f}, n={len(combined)}")
        print(f"{b:20s}:  n={len(combined)}")


In [41]:
concat_pb_analysis('bleu1')

MedQA               :  n=8903
MedExpQA            :  n=875
USMLE_STEP_1        :  n=656
USMLE_STEP_2        :  n=763
USMLE_STEP_3        :  n=853


In [ ]:

for b in benchmarks:
    print(f"\n=== {b} ===")
    rows = []
    for m in models:
        p = f"{baseline_response_path}/{m}_{b}_overlap.csv"
        if not os.path.exists(p): continue
        df = pd.read_csv(p)[['acc', 'bleu4']].copy()
        df['acc'] = df['acc'].astype(int)
        correct   = df[df['acc'] == 1]['bleu4'].mean()
        incorrect = df[df['acc'] == 0]['bleu4'].mean()
        rows.append({'model': m, 'correct': round(correct,4), 'incorrect': round(incorrect,4),
                     'diff': round(correct - incorrect, 4)})
    print(pd.DataFrame(rows).to_string(index=False))


=== USMLE_STEP_1 ===
      model  correct  incorrect    diff
0.5B-Qwen25   0.0035     0.0073 -0.0038
1.5B-Qwen25   0.0052     0.0073 -0.0022
  3B-Qwen25   0.0096     0.0087  0.0009
  7B-Qwen25   0.0068     0.0054  0.0014
 14B-Qwen25   0.0052     0.0047  0.0005
 32B-Qwen25   0.0056     0.0033  0.0023
 72B-Qwen25   0.0044     0.0033  0.0011

=== USMLE_STEP_2 ===
      model  correct  incorrect    diff
0.5B-Qwen25   0.0093     0.0089  0.0004
1.5B-Qwen25   0.0084     0.0075  0.0009
  3B-Qwen25   0.0097     0.0105 -0.0008
  7B-Qwen25   0.0073     0.0088 -0.0015
 14B-Qwen25   0.0078     0.0076  0.0001
 32B-Qwen25   0.0075     0.0069  0.0006
 72B-Qwen25   0.0078     0.0085 -0.0007

=== USMLE_STEP_3 ===
      model  correct  incorrect    diff
0.5B-Qwen25   0.0058     0.0080 -0.0021
1.5B-Qwen25   0.0087     0.0074  0.0013
  3B-Qwen25   0.0112     0.0097  0.0015
  7B-Qwen25   0.0076     0.0072  0.0004
 14B-Qwen25   0.0082     0.0080  0.0001
 32B-Qwen25   0.0081     0.0078  0.0002
 72B-Qwen25   

In [37]:
for b in benchmarks:
    print(f"\n=== {b} ===")
    rows = []
    for m in models:
        jp = f"{json_base}/{m}_{b}_q_square.json"
        if not os.path.exists(jp): continue
        df = pd.DataFrame(load_json_safe(jp))[['acc', 'background_recall', 'ques_recall']].dropna()
        df['acc'] = df['acc'].astype(int)
        for metric in ['background_recall', 'ques_recall']:
            correct   = df[df['acc'] == 1][metric].mean()
            incorrect = df[df['acc'] == 0][metric].mean()
            rows.append({'model': m, 'metric': metric,
                         'correct': round(correct, 4),
                         'incorrect': round(incorrect, 4),
                         'diff': round(correct - incorrect, 4)})
    print(pd.DataFrame(rows).to_string(index=False))



=== MedQA ===
      model            metric  correct  incorrect    diff
0.5B-Qwen25 background_recall   0.2076     0.2163 -0.0087
0.5B-Qwen25       ques_recall   0.5706     0.5779 -0.0073
1.5B-Qwen25 background_recall   0.1970     0.2021 -0.0050
1.5B-Qwen25       ques_recall   0.5627     0.5623  0.0004
  3B-Qwen25 background_recall   0.2398     0.2357  0.0041
  3B-Qwen25       ques_recall   0.7055     0.6828  0.0227
  7B-Qwen25 background_recall   0.2522     0.2401  0.0121
  7B-Qwen25       ques_recall   0.6816     0.6437  0.0380
 14B-Qwen25 background_recall   0.2406     0.2303  0.0104
 14B-Qwen25       ques_recall   0.6392     0.6504 -0.0112
 32B-Qwen25 background_recall   0.2660     0.2483  0.0178
 32B-Qwen25       ques_recall   0.6626     0.6583  0.0043
 72B-Qwen25 background_recall   0.2936     0.2885  0.0050
 72B-Qwen25       ques_recall   0.6972     0.6969  0.0003

=== MedExpQA ===
      model            metric  correct  incorrect    diff
0.5B-Qwen25 background_recall   0.2933 

In [13]:
for b in benchmarks:
    print(f"\n=== {b} ===")
    rows = []
    for m in models:
        p = f"{baseline_response_path}/{m}_{b}_overlap.csv"
        if not os.path.exists(p): continue
        df = pd.read_csv(p)[['acc', 'explanation']].copy()
        df['acc'] = df['acc'].astype(int)
        df['length'] = df['explanation'].astype(str).apply(lambda x: len(x.split()))
        correct   = df[df['acc'] == 1]['length'].mean()
        incorrect = df[df['acc'] == 0]['length'].mean()
        overall   = df['length'].mean()
        rows.append({'model': m, 'overall': round(overall,1),
                     'correct': round(correct,1), 'incorrect': round(incorrect,1)})
    print(pd.DataFrame(rows).to_string(index=False))


=== USMLE_STEP_1 ===
      model  overall  correct  incorrect
0.5B-Qwen25    201.1    191.0      202.9
1.5B-Qwen25    232.8    219.6      240.6
  3B-Qwen25    276.0    276.8      275.4
  7B-Qwen25    304.4    298.2      313.1
 14B-Qwen25    287.9    278.9      309.0
 32B-Qwen25    296.5    290.7      319.6
 72B-Qwen25    365.4    358.3      391.8

=== USMLE_STEP_2 ===
      model  overall  correct  incorrect
0.5B-Qwen25    260.4    264.4      258.9
1.5B-Qwen25    265.7    271.3      262.1
  3B-Qwen25    312.5    314.4      310.3
  7B-Qwen25    313.1    304.4      327.1
 14B-Qwen25    307.5    305.4      312.9
 32B-Qwen25    304.8    304.4      306.6
 72B-Qwen25    373.5    373.5      373.6

=== USMLE_STEP_3 ===
      model  overall  correct  incorrect
0.5B-Qwen25    266.4    264.4      266.9
1.5B-Qwen25    260.3    251.3      267.7
  3B-Qwen25    310.5    307.8      313.5
  7B-Qwen25    313.2    307.8      325.1
 14B-Qwen25    306.0    303.9      315.3
 32B-Qwen25    309.3    305.8   

In [26]:
def load_json_safe(path):
    with open(path, 'r') as f:
        text = f.read()
    text = re.sub(r'\bNaN\b', 'null', text)
    text = re.sub(r'\bInfinity\b', 'null', text)
    text = re.sub(r'\b-Infinity\b', 'null', text)
    return json.loads(text)

In [31]:

def load_json_safe(path):
    with open(path, 'r') as f:
        text = f.read()
    text = re.sub(r'\bNaN\b', 'null', text)
    text = re.sub(r'\bInfinity\b', 'null', text)
    text = re.sub(r'\b-Infinity\b', 'null', text)
    text = re.sub(r':\s*,', ': null,', text)   # 新加这行
    return json.loads(text)

models = ['0.5B-Qwen25','1.5B-Qwen25','3B-Qwen25','7B-Qwen25','14B-Qwen25','32B-Qwen25','72B-Qwen25']
benchmarks = ['MedQA','MedExpQA','USMLE_STEP_1','USMLE_STEP_2','USMLE_STEP_3']

json_base = "/projectnb/vkolagrp/yiliu/QA_pipeline/Results/q_square_metric/LEM_biomedicalNER"
csv_base  = "/projectnb/vkolagrp/yiliu/QA_pipeline/Results/baseline_metric"

baseline_metrics = ['bleu1', 'bleu4', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1']
my_metrics       = ['background_recall', 'ques_recall']

rows = []
for b in benchmarks:
    json_dfs, csv_dfs = [], []
    for m in models:
        jp = f"{json_base}/{m}_{b}_q_square.json"
        cp = f"{csv_base}/{m}_{b}_overlap.csv"
        if os.path.exists(jp):
            jdf = pd.DataFrame(load_json_safe(jp))[['acc'] + my_metrics]
            jdf['acc'] = jdf['acc'].astype(int)
            jdf = jdf.dropna(subset=my_metrics)
            json_dfs.append(jdf)
        if os.path.exists(cp):
            cdf = pd.read_csv(cp)[['acc'] + baseline_metrics]
            cdf['acc'] = cdf['acc'].astype(int)
            csv_dfs.append(cdf)

    if not json_dfs or not csv_dfs:
        continue

    jcombined = pd.concat(json_dfs, ignore_index=True)
    ccombined = pd.concat(csv_dfs, ignore_index=True)

    row = {'benchmark': b}
    for m in my_metrics:
        r, p = pointbiserialr(jcombined['acc'], jcombined[m])
        row[m] = f"{r:.4f}{'**' if p<0.01 else ('*' if p<0.05 else '')}"
    for m in baseline_metrics:
        r, p = pointbiserialr(ccombined['acc'], ccombined[m])
        row[m] = f"{r:.4f}{'**' if p<0.01 else ('*' if p<0.05 else '')}"
    rows.append(row)

result = pd.DataFrame(rows).set_index('benchmark')
print(result.to_string())

             background_recall ques_recall      bleu1     bleu4     meteor     rouge1     rouge2     rougeL bertscore_f1
benchmark                                                                                                               
MedQA                 0.0595**    0.0373**  -0.0593**    0.0113   -0.0242*  -0.0667**    -0.0028  -0.0425**    -0.0826**
MedExpQA              0.0917**      0.0223    -0.0068  0.0987**     0.0273    -0.0050     0.0532     0.0206      -0.0435
USMLE_STEP_1            0.0129      0.0055   -0.0977*   -0.0389    -0.0533  -0.1105**    -0.0596  -0.1050**    -0.1287**
USMLE_STEP_2            0.0394     -0.0236  -0.1153**   -0.0327  -0.1373**  -0.1517**  -0.1092**  -0.1434**    -0.1018**
USMLE_STEP_3          0.0900**     0.0729*     0.0071    0.0081    -0.0005    -0.0113    -0.0011    -0.0171      -0.0639


In [ ]:
rows = []
for metric in my_metrics + baseline_metrics:
    row = {'metric': metric}
    for m in models:
        for b in :
            if metric in my_metrics:
                jp = f"{json_base}/{m}_{b}_q_square.json"
                if not os.path.exists(jp): continue
                df = pd.DataFrame(load_json_safe(jp))[['acc', metric]].dropna()
            else:
                cp = f"{csv_base}/{m}_{b}_overlap.csv"
                if not os.path.exists(cp): continue
                df = pd.read_csv(cp)[['acc', metric]]
            df['acc'] = df['acc'].astype(int)
            r, p = pointbiserialr(df['acc'], df[metric])
            stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            row[f"{m}\n{b}"] = f"{r:.4f}{stars}"
    rows.append(row)

result = pd.DataFrame(rows).set_index('metric')
print(result.to_string())

                  0.5B-Qwen25\nMedQA 0.5B-Qwen25\nMedExpQA 0.5B-Qwen25\nUSMLE_STEP_1 0.5B-Qwen25\nUSMLE_STEP_2 0.5B-Qwen25\nUSMLE_STEP_3 1.5B-Qwen25\nMedQA 1.5B-Qwen25\nMedExpQA 1.5B-Qwen25\nUSMLE_STEP_1 1.5B-Qwen25\nUSMLE_STEP_2 1.5B-Qwen25\nUSMLE_STEP_3 3B-Qwen25\nMedQA 3B-Qwen25\nMedExpQA 3B-Qwen25\nUSMLE_STEP_1 3B-Qwen25\nUSMLE_STEP_2 3B-Qwen25\nUSMLE_STEP_3 7B-Qwen25\nMedQA 7B-Qwen25\nMedExpQA 7B-Qwen25\nUSMLE_STEP_1 7B-Qwen25\nUSMLE_STEP_2 7B-Qwen25\nUSMLE_STEP_3 14B-Qwen25\nMedQA 14B-Qwen25\nMedExpQA 14B-Qwen25\nUSMLE_STEP_1 14B-Qwen25\nUSMLE_STEP_2 14B-Qwen25\nUSMLE_STEP_3 32B-Qwen25\nMedQA 32B-Qwen25\nMedExpQA 32B-Qwen25\nUSMLE_STEP_1 32B-Qwen25\nUSMLE_STEP_2 32B-Qwen25\nUSMLE_STEP_3 72B-Qwen25\nMedQA 72B-Qwen25\nMedExpQA 72B-Qwen25\nUSMLE_STEP_1 72B-Qwen25\nUSMLE_STEP_2 72B-Qwen25\nUSMLE_STEP_3
metric                                                                                                                                                                                  

In [35]:
for metric in my_metrics + baseline_metrics:
    rows = []
    for m in models:
        row = {'model': m}
        for b in benchmarks:
            if metric in my_metrics:
                jp = f"{json_base}/{m}_{b}_q_square.json"
                if not os.path.exists(jp): continue
                df = pd.DataFrame(load_json_safe(jp))[['acc', metric]].dropna()
            else:
                cp = f"{csv_base}/{m}_{b}_overlap.csv"
                if not os.path.exists(cp): continue
                df = pd.read_csv(cp)[['acc', metric]]
            df['acc'] = df['acc'].astype(int)
            r, p = pointbiserialr(df['acc'], df[metric])
            stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            row[b] = f"{r:.4f}{stars}"
        rows.append(row)

    df_out = pd.DataFrame(rows).set_index('model')
    df_out.to_csv(f"/projectnb/vkolagrp/yiliu/QA_pipeline/Result_rebuttal/result_clarification/pb_table_{metric}.csv")
    print(f"\n=== {metric} ===")
    print(df_out.to_string())


=== background_recall ===
               MedQA MedExpQA USMLE_STEP_1 USMLE_STEP_2 USMLE_STEP_3
model                                                               
0.5B-Qwen25  -0.0283  -0.0490      -0.0488       0.0166      -0.0059
1.5B-Qwen25  -0.0173   0.0509      -0.1420      -0.0167      -0.0215
3B-Qwen25     0.0143   0.0173      -0.0842      -0.0258      -0.1383
7B-Qwen25     0.0422   0.1495       0.0277      -0.0388      -0.0068
14B-Qwen25    0.0349   0.0690       0.0217      -0.0836       0.1258
32B-Qwen25   0.0570*  -0.0549      -0.0139       0.0081       0.0907
72B-Qwen25    0.0142   0.0318      -0.0377       0.0430       0.1095

=== ques_recall ===
               MedQA  MedExpQA USMLE_STEP_1 USMLE_STEP_2 USMLE_STEP_3
model                                                                
0.5B-Qwen25  -0.0093    0.0202      -0.1916       0.0839      -0.0356
1.5B-Qwen25   0.0006   -0.0148      -0.0041      -0.1094       0.0487
3B-Qwen25     0.0342   -0.0096       0.0652     -0.

In [36]:
rows = []
for m in models:
    row = {'model': m}
    for b in benchmarks:
        cp = f"{csv_base}/{m}_{b}_overlap.csv"
        if not os.path.exists(cp): continue
        df = pd.read_csv(cp)
        df['acc'] = df['acc'].astype(int)
        row[b] = round(df['acc'].mean(), 4)
    rows.append(row)

acc_df = pd.DataFrame(rows).set_index('model')
print(acc_df.to_string())

              MedQA  MedExpQA  USMLE_STEP_1  USMLE_STEP_2  USMLE_STEP_3
model                                                                  
0.5B-Qwen25  0.3162     0.288        0.1489        0.2661        0.1967
1.5B-Qwen25  0.4713     0.448        0.3696        0.3945        0.4508
3B-Qwen25    0.4949     0.488        0.4574        0.5505        0.5207
7B-Qwen25    0.5899     0.600        0.5851        0.6147        0.6885
14B-Qwen25   0.6599     0.648        0.7021        0.7156        0.8197
32B-Qwen25   0.7078     0.784        0.7979        0.7982        0.8361
72B-Qwen25   0.7408     0.824        0.7872        0.8349        0.8443
